# 🎵 MÓDULO 1 — Análisis del Mercado Musical
**Proyecto:** ProyectoFinal23  
**Fuente principal:** Last.fm API — https://www.last.fm/api  
**Objetivo:** Entender patrones de popularidad del mercado musical y construir la base analítica para predecir hits.

---

### 🗓️ Historial de trabajo
| Semana | Qué se hizo |
|--------|-------------|
| Semana 1 | Definición del proyecto. Lluvia de ideas. Primer contacto con Last.fm API. |
| Semana 2 | Recolección multi-endpoint. Problema de datos insuficientes. Prueba de AcousticBrainz (descartada ~98% NaN). |
| Semana 3 | Reestructuración del pipeline. Nuevos endpoints según guía oficial de la API. Enriquecimiento de genre_tag con `track.getTopTags`. |

---

### 📋 Índice
1. [Setup e imports](#setup)
2. [Configuración de la API](#config)
3. [Recolección de datos — Pipeline multi-endpoint](#data)
   - [3.1 Función base: fetch_lastfm](#fetch)
   - [3.2 chart.getTopTracks — Top global](#chart)
   - [3.3 chart.getTopArtists — Artistas globales](#chartartists)
   - [3.4 geo.getTopTracks — Top por país](#geo)
   - [3.5 geo.getTopArtists — Artistas por país](#geoartists)
   - [3.6 track.getTopTags — Enriquecimiento de géneros](#tags)
   - [3.7 tag.getWeeklyChartList — Evolución temporal](#weekly)
   - [3.8 Construcción del dataset final](#build)
4. [Limpieza y Feature Engineering](#clean)
5. [EDA — Análisis del mercado musical](#eda)
   - [5.1 Top canciones y artistas](#top)
   - [5.2 Evolución temporal de popularidad](#temporal)
   - [5.3 Géneros: crecimiento vs decrecimiento](#generos)
   - [5.4 Análisis geográfico](#geo-eda)
   - [5.5 Correlaciones y heatmap](#correlaciones)
6. [Resumen de insights](#resumen)

---
### ⚠️ Nota importante sobre Last.fm vs Spotify

* Problema Semana 3: Falta añadir metada de las canciones, conseguir a través de IA pasando temas manualmente?

**Para los modelos ML**: usaremos `playcount` y `listeners` como variables objetivo (target), y los tags/géneros/país como variables independientes.

---
## 1. Setup e imports <a id='setup'></a>

In [2]:
# ── Librerías estándar ────────────────────────────────────────────────────────
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ── Datos ─────────────────────────────────────────────────────────────────────
import requests
import numpy as np
import pandas as pd

# ── Visualización ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Configuración de plots ────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

print('Imports correctos')

Imports correctos


---
## 2. Configuración de la API <a id='config'></a>

**Buena práctica:** la API key no debe subirse a GitHub.  
La opción correcta es guardarla en un archivo `.env` en la raíz del proyecto y cargarlo con `python-dotenv`.

**Cómo crear el .env correctamente:**
```
# 1. Crear archivo .env en la raíz del proyecto (al mismo nivel que este notebook)
# 2. Contenido del .env:
LASTFM_API_KEY='tu_api_key_aqui'

# 3. Asegurarse de que .gitignore incluye:
.env
*.csv
checkpoint_*.csv
```

**Tabla de errores HTTP frecuentes:**

| Código | Significado | Quién tiene el problema |
|--------|-------------|-------------------------|
| 403    | Forbidden   | ❌ tú (API key incorrecta o sin permisos) |
| 429    | Rate Limit  | ⚠️ tú (demasiadas peticiones por segundo) |
| 502    | Bad Gateway | ❌ servidor Last.fm (reintentar) |

#### *Teoria: Función general para hacerpeticiones API con retry y backoff exponencial.*





**Objetivo:** Realizar peticiones API evitando errores de rate limits, de servidor y problemas de red.
* APY_KEY:
* BASE_URL:
* DELAY: tiempo de espera entre peticiones. Last.fm permite ~5 req/seg. Con 0.25s hacemos 4 req/seg → margen de seguridad.

**Parámetros:**

* params   — dict con los parámetros de la petición (method, page, etc.).
    * base_params = {'api_key': API_KEY, 'format': 'json'}
        * api_key → autenticación
        * format=json → respuesta en formato JSON
    * base_params.update(params), se añaden los parámetros específicos de cada llamada.

* retries  — número máximo de reintentos ante errores de servidor. 

**Bucle:** 

* attempt — número de intentos por petición de cada dato, se usa para contar intentos y calcular el tiempo de espera (backoff).

* try: requests.get(BASE_URL, params=base_params, timeout=10) — Evita que el programa se quede bloqueado.
        - BASE_URL, endpoint de Last.fm.
        * params, parámetros de la API.
        * timeout=10, máximo 10 segundos de espera.

**Condición del bucle:** Para resolver los "server-side errors", errores del servidor al hacer las peticiones.

* **Manejo de errores:** response.status_code in [429, 500, 502, 503, 504]
    * 429, demasiadas peticiones (rate limit).
    * 500–504, errores del servidor.

    * **Backoff exponencial:** wait = 2 ** attempt & time.sleep(wait) marcan el tiempo de espera despues de cada intento de petición permitiendo al servidor recuperarse.
    * continue, salta al siguiente intento del bucle sin romper / parar la función.

* Retorna el JSON de la respuesta o None si falla definitivamente.
    * response.raise_for_status(), es la validación de la respuesta. Si hay error HTTP "lanza expedicion" DUDA si no continua.
    * return response.json(), convierte la respuesta en diccionario Python y devuelve los datos de la API.

* **Manejo de errores en la red:** * except requests.exceptions.RequestException as e — captura errores de timeout, desconexión o DNS error. Se utiliza el mismo sitmea backoff exponencial que en los errores de servidor para resolver los problemas.
     

* Si todos los intentos fallas aparece un mensaje aparece un mensaje con el numero de intentos y devuelve None como dato.

Conceptos clave de la API:

- **Throttling** → ir lento a propósito para no superar el rate limit
- **Backoff exponencial** → esperar 2s, 4s, 8s... si hay error
- **Paginación** → iterar por páginas para obtener todos los datos
- **Deduplicación** → eliminar duplicados entre endpoints
- **Timeout** → evitar bloqueos
- **HTTP status codes** → interpretar respuestas

#### *Código:*

In [3]:
# ── API Key ─────────────────────────────────────────────────────────────────── DUDA : FALTA HACER
# Opción 1 (recomendada): cargar desde variable de entorno
# API_KEY = os.getenv('LASTFM_API_KEY')


In [4]:
API_KEY = '63e059c3c912a3f642daf2372484d183'
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
DELAY = 0.25

print('Configuración lista')
print(f'API KEY cargada: {"✅" if API_KEY else "❌ — revisa tu .env"}')

Configuración lista
API KEY cargada: ✅


---
## 3. Recolección de datos — Pipeline multi-endpoint <a id='data'></a>

**Por qué usamos múltiples endpoints:**  
Cada endpoint de Last.fm devuelve datos distintos. Combinarlos nos permite llegar a los 60.000 registros requeridos y tener datos de calidad (sin NaN en campos clave).

| Endpoint | Qué devuelve | Para qué lo usamos |
|---|---|---|
| `chart.getTopTracks` | Top tracks globales (paginado) | Popularidad global, playcount, listeners |
| `chart.getTopArtists` | Top artistas globales | Popularidad de artistas |
| `geo.getTopTracks` | Top tracks **por país** | Análisis geográfico |
| `geo.getTopArtists` | Top artistas por país | Artistas dominantes por región |
| `track.getTopTags` | Tags (géneros) de cada track | Enriquece genre_tag (antes era todo UNKNOWN) |
| `tag.getWeeklyChartList` | Lista de charts semanales disponibles | Análisis de evolución temporal |

**Conceptos clave:**
- **Throttling** → ir lento a propósito para no superar el rate limit
- **Backoff exponencial** → esperar 2s, 4s, 8s... si hay error del servidor
- **Paginación** → iterar por páginas para obtener todos los datos
- **Deduplicación** → eliminar duplicados entre endpoints al combinar

---
### 3.1 Función base: fetch_lastfm <a id='fetch'></a>

**¿Para qué sirve?**  
Es la función que usamos para hacer **cualquier petición** a la API de Last.fm.  
Es genérica para no repetir código — todos los demás endpoints la llaman.

**Parámetros:**
- `params` — diccionario con los parámetros específicos de cada llamada (qué método, qué página, etc.)
- `retries` — cuántas veces reintentamos si hay error del servidor

**Cómo funciona el retry con backoff:**
```
Intento 1 → falla → espera 2s (2^1)
Intento 2 → falla → espera 4s (2^2)
Intento 3 → falla → espera 8s (2^3)
Intento 4 → falla → espera 16s (2^4)
Intento 5 → falla → devuelve None
```
La espera crece exponencialmente para dar tiempo al servidor a recuperarse.

In [5]:
def fetch_lastfm(params, retries=5):
    """
    Hace una petición a la API de Last.fm con retry y backoff exponencial.
    
    Parámetros:
        params   — dict con los parámetros específicos de la petición
        retries  — número máximo de reintentos si el servidor falla
    
    Devuelve el JSON de la respuesta, o None si falla definitivamente.
    """
    # Parámetros base que toda petición a Last.fm necesita
    base_params = {'api_key': API_KEY, 'format': 'json'}
    
    # Añadimos los parámetros específicos de cada llamada encima de los base
    base_params.update(params)

    for attempt in range(retries):
        try:
            # timeout=10: si el servidor no responde en 10s, lanzamos error
            response = requests.get(BASE_URL, params=base_params, timeout=10)

            # Errores del servidor (5xx) o rate limit (429) → reintentamos
            # Nota: 4xx son errores nuestros (key, permisos), 5xx son del servidor
            if response.status_code in [429, 500, 502, 503, 504]:
                wait = 2 ** attempt  # backoff exponencial: 1s, 2s, 4s, 8s...
                print(f'  ⚠️  HTTP {response.status_code} → retry en {wait}s (intento {attempt+1}/{retries})')
                time.sleep(wait)
                continue  # vuelve al inicio del bucle sin romperlo

            # raise_for_status(): si hay cualquier otro error HTTP, lanza excepción
            response.raise_for_status()
            return response.json()  # convierte la respuesta en diccionario Python

        except requests.exceptions.RequestException as e:
            # Errores de red: timeout, DNS, conexión caída, etc.
            wait = 2 ** attempt
            print(f'  ⚠️  Error de red: {e} → retry en {wait}s')
            time.sleep(wait)

    print(f'  ❌ Fallo definitivo tras {retries} intentos')
    return None  # devolvemos None para que el código que llama a esta función lo maneje

---
### 3.2 chart.getTopTracks — Top global <a id='chart'></a>

**¿Qué devuelve?**  
Los tracks más escuchados globalmente en el momento de la consulta. Es paginado: cada página tiene hasta 200 tracks.

**¿Por qué este endpoint?**  
Es la fuente principal de popularidad global. Garantiza que todos los tracks tienen `playcount` y `listeners` reales (no vacíos).  
A diferencia de `track.search`, `chart.getTopTracks` solo devuelve tracks con actividad real de usuarios.

**Estructura de la respuesta:**
```json
{
  "tracks": {
    "track": [
      { "name": "Espresso", "playcount": "54000000", "listeners": "2100000",
        "duration": "175", "mbid": "...",
        "artist": {"name": "Sabrina Carpenter"} }
    ]
  }
}
```

In [6]:
def collect_chart_tracks(n_pages=150, limit=200):
    """
    Recoge los tracks más populares del ranking global (chart.getTopTracks).
    
    Parámetros:
        n_pages — cuántas páginas pedir (150 páginas x 200 tracks = ~30.000 tracks)
        limit   — tracks por página (máximo 200 en Last.fm)
    
    Devuelve una lista de diccionarios, uno por track.
    """
    tracks = []
    
    for page in range(1, n_pages + 1):
        # Llamamos a la función base con los parámetros específicos de este endpoint
        data = fetch_lastfm({'method': 'chart.getTopTracks', 'page': page, 'limit': limit})
        
        if not data:
            break  # si fetch_lastfm devuelve None, paramos
        
        # Navegamos por la estructura JSON de la respuesta
        page_tracks = data.get('tracks', {}).get('track', [])
        
        if not page_tracks:
            break  # si la página está vacía, no hay más datos
        
        for t in page_tracks:
            # t.get('artist', {}) → el artista viene como diccionario anidado
            artist_name = t.get('artist', {}).get('name', '') if isinstance(t.get('artist'), dict) else str(t.get('artist', ''))
            
            tracks.append({
                'name'       : t.get('name', ''),
                'artist'     : artist_name,
                'playcount'  : t.get('playcount', 0),
                'listeners'  : t.get('listeners', 0),
                'duration'   : t.get('duration', 0),
                'mbid'       : t.get('mbid', ''),
                'country'    : 'GLOBAL',   # este endpoint es global, no tiene país
                'genre_tag'  : 'UNKNOWN',  # se enriquecerá después con track.getTopTags
                'rank_global': (page - 1) * limit + page_tracks.index(t) + 1,
                'rank_by_country': None,
                'source_endpoint': 'chart.getTopTracks',
            })
        
        time.sleep(DELAY)  # throttling: evitar superar el rate limit
        
        if page % 25 == 0:
            print(f'  chart.getTopTracks → página {page}/{n_pages} | acumulados: {len(tracks):,}')
    
    print(f'  ✅ chart.getTopTracks completado: {len(tracks):,} tracks')
    return tracks

---
### 3.3 chart.getTopArtists — Artistas globales <a id='chartartists'></a>

**¿Qué devuelve?**  
Los artistas más escuchados globalmente. Nos da `listeners` y `playcount` a nivel de artista (no de canción individual).  

**¿Para qué lo usamos en el EDA?**  
Complementa `chart.getTopTracks`: con el top de tracks sabemos qué canciones triunfan, con el top de artistas sabemos qué **artistas** dominan el mercado independientemente de qué canción sea.

**Estructura de la respuesta:**
```json
{
  "artists": {
    "artist": [
      { "name": "Taylor Swift", "playcount": "500000000", "listeners": "8000000" }
    ]
  }
}
```

In [7]:
def collect_chart_artists(n_pages=50, limit=200):
    """
    Recoge los artistas más populares del ranking global (chart.getTopArtists).
    
    Parámetros:
        n_pages — páginas a recoger (50 x 200 = ~10.000 artistas)
        limit   — artistas por página
    
    Devuelve una lista de diccionarios, uno por artista.
    """
    artists = []
    
    for page in range(1, n_pages + 1):
        data = fetch_lastfm({'method': 'chart.getTopArtists', 'page': page, 'limit': limit})
        
        if not data:
            break
        
        page_artists = data.get('artists', {}).get('artist', [])
        
        if not page_artists:
            break
        
        for a in page_artists:
            artists.append({
                'artist'          : a.get('name', ''),
                'artist_playcount': a.get('playcount', 0),
                'artist_listeners': a.get('listeners', 0),
                'mbid'            : a.get('mbid', ''),
                'rank_global_artist': (page - 1) * limit + page_artists.index(a) + 1,
            })
        
        time.sleep(DELAY)
        
        if page % 10 == 0:
            print(f'  chart.getTopArtists → página {page}/{n_pages} | acumulados: {len(artists):,}')
    
    print(f'  ✅ chart.getTopArtists completado: {len(artists):,} artistas')
    return artists

---
### 3.4 geo.getTopTracks — Top por país <a id='geo'></a>

**¿Por qué este endpoint para el análisis geográfico?**  
La guía de la API recomienda esto explícitamente:  
> *"No intentes inferir la localización de una canción global; utiliza los endpoints de `geo` para construir datasets por país."*

Al pedirle directamente a Last.fm los tracks de España, Francia, etc., los datos llegan con el `country` ya asignado y sin NaN en ese campo.  
Esto resuelve el problema anterior donde `genre_tag` y `country` eran todo UNKNOWN para los tracks globales.

**Parámetro `country`:** debe ser el nombre del país en inglés (ISO 3166-1), ej: `'spain'`, `'france'`.

In [8]:
def collect_geo_tracks(countries, pages_per_country=10, limit=200):
    """
    Recoge los tracks más populares por país (geo.getTopTracks).
    
    Parámetros:
        countries          — lista de países en inglés (ej: ['spain', 'france'])
        pages_per_country  — páginas por país (10 x 200 = 2.000 tracks por país)
        limit              — tracks por página
    
    Devuelve una lista de diccionarios, uno por track.
    Ventaja clave: el campo 'country' ya viene asignado, no hay NaN.
    """
    tracks = []
    
    for country in countries:
        country_tracks_before = len(tracks)  # para saber cuántos añadimos de este país
        
        for page in range(1, pages_per_country + 1):
            data = fetch_lastfm({
                'method' : 'geo.getTopTracks',
                'country': country,  # parámetro obligatorio de este endpoint
                'page'   : page,
                'limit'  : limit
            })
            
            if not data:
                break
            
            page_tracks = data.get('tracks', {}).get('track', [])
            
            if not page_tracks:
                break
            
            for t in page_tracks:
                artist_name = t.get('artist', {}).get('name', '') if isinstance(t.get('artist'), dict) else str(t.get('artist', ''))
                
                tracks.append({
                    'name'           : t.get('name', ''),
                    'artist'         : artist_name,
                    'playcount'      : t.get('listeners', 0),  # geo endpoint devuelve 'listeners' no 'playcount'
                    'listeners'      : t.get('listeners', 0),
                    'duration'       : t.get('duration', 0),
                    'mbid'           : t.get('mbid', ''),
                    'country'        : country,  # ya sabemos el país, sin NaN
                    'genre_tag'      : 'UNKNOWN',  # se enriquecerá con track.getTopTags
                    'rank_global'    : None,
                    'rank_by_country': (page - 1) * limit + page_tracks.index(t) + 1,
                    'source_endpoint': 'geo.getTopTracks',
                })
            
            time.sleep(DELAY)
        
        n_added = len(tracks) - country_tracks_before
        print(f'  geo.getTopTracks [{country}] → {n_added:,} tracks')
    
    print(f'  ✅ geo.getTopTracks completado: {len(tracks):,} tracks totales')
    return tracks

---
### 3.5 geo.getTopArtists — Artistas por país <a id='geoartists'></a>

**¿Qué devuelve?**  
Los artistas más escuchados en cada país. Nos permite comparar qué artistas dominan en España vs México vs Japón, detectando diferencias culturales.

**Uso en el EDA:** construiremos una tabla artistas × países para ver qué artistas son globales vs locales.

In [9]:
def collect_geo_artists(countries, pages_per_country=5, limit=200):
    """
    Recoge los artistas más populares por país (geo.getTopArtists).
    
    Parámetros:
        countries          — lista de países
        pages_per_country  — páginas por país
        limit              — artistas por página
    """
    artists = []
    
    for country in countries:
        country_before = len(artists)
        
        for page in range(1, pages_per_country + 1):
            data = fetch_lastfm({
                'method' : 'geo.getTopArtists',
                'country': country,
                'page'   : page,
                'limit'  : limit
            })
            
            if not data:
                break
            
            page_artists = data.get('topartists', {}).get('artist', [])
            
            if not page_artists:
                break
            
            for a in page_artists:
                artists.append({
                    'artist'           : a.get('name', ''),
                    'artist_listeners' : a.get('listeners', 0),
                    'mbid'             : a.get('mbid', ''),
                    'country'          : country,
                    'rank_by_country_artist': (page - 1) * limit + page_artists.index(a) + 1,
                })
            
            time.sleep(DELAY)
        
        print(f'  geo.getTopArtists [{country}] → {len(artists) - country_before:,} artistas')
    
    print(f'  ✅ geo.getTopArtists completado: {len(artists):,} artistas totales')
    return artists

---
### 3.6 track.getTopTags — Enriquecimiento de géneros <a id='tags'></a>

**Este es el paso clave para resolver el problema de `genre_tag = UNKNOWN`.**

**¿Por qué teníamos todo UNKNOWN?**  
Los endpoints `chart.getTopTracks` y `geo.getTopTracks` no devuelven información de género.  
Para obtener el género de cada track, hay que llamar **por separado** a `track.getTopTags`.

**¿Cómo funciona `track.getTopTags`?**  
Last.fm no tiene géneros fijos — los géneros son **tags** (etiquetas) creadas por la comunidad de usuarios.  
Cada canción tiene múltiples tags. Los más votados son los más representativos.  
Nosotros nos quedamos con el **tag principal** (el de mayor count) como género.

**Estructura de la respuesta:**
```json
{
  "toptags": {
    "tag": [
      {"name": "pop", "count": 100},
      {"name": "indie", "count": 75},
      {"name": "2010s", "count": 50}
    ]
  }
}
```

**Optimización:**  
No pedimos tags para TODOS los 60.000 tracks (serían 60.000 peticiones extra, ~4 horas).  
Solo enriquecemos los tracks únicos que tenemos — muchos tracks se repiten en el dataset multi-endpoint.

In [10]:
def get_track_genre(track_name, artist_name):
    """
    Obtiene el género principal de un track usando track.getTopTags.
    
    Parámetros:
        track_name  — nombre de la canción
        artist_name — nombre del artista
    
    Devuelve el nombre del tag más popular como string, o 'UNKNOWN' si no hay datos.
    
    Nota: usamos 'autocorrect=1' para que Last.fm corrija pequeños errores
    de escritura en el nombre del track (typos frecuentes en el dataset).
    """
    data = fetch_lastfm({
        'method'     : 'track.getTopTags',
        'track'      : track_name,
        'artist'     : artist_name,
        'autocorrect': 1,  # Last.fm corrige typos automáticamente
    })
    
    if not data:
        return 'UNKNOWN'
    
    tags = data.get('toptags', {}).get('tag', [])
    
    if not tags:
        return 'UNKNOWN'
    
    # Los tags vienen ordenados por count (más votado primero)
    # Filtramos tags que sean géneros reales (evitamos tags tipo '2010s', 'seen live', etc.)
    TAGS_A_IGNORAR = {'seen live', 'favorites', 'love', 'favourite', 'awesome',
                      'cool', 'beautiful', '00s', '90s', '80s', '70s', '10s',
                      '2000s', '2010s', '2020s', 'good', 'nice', 'great'}
    
    for tag in tags:
        tag_name = tag.get('name', '').lower().strip()
        if tag_name and tag_name not in TAGS_A_IGNORAR:
            return tag_name  # devolvemos el primer tag que sea un género real
    
    return 'UNKNOWN'

In [11]:
def enrich_with_genres(df, checkpoint_path='checkpoint_genres.csv'):
    """
    Enriquece el DataFrame añadiendo el género (tag principal) a cada track.
    
    Parámetros:
        df              — DataFrame con columnas 'name', 'artist', 'genre_tag'
        checkpoint_path — ruta donde guardar el progreso (para reanudar si se interrumpe)
    
    Devuelve el DataFrame con 'genre_tag' actualizado.
    
    Optimización: solo pide tags para tracks únicos (evita peticiones duplicadas).
    Checkpoint: guarda el progreso cada 500 tracks para no perder trabajo.
    """
    df = df.copy()
    
    # Solo enriquecemos los que aún tienen UNKNOWN
    mask_unknown = df['genre_tag'] == 'UNKNOWN'
    tracks_to_enrich = df[mask_unknown][['name', 'artist']].drop_duplicates()
    
    print(f'Tracks a enriquecer: {len(tracks_to_enrich):,} (únicos con genre_tag=UNKNOWN)')
    
    # Diccionario para guardar los resultados: clave = (name, artist), valor = género
    genre_cache = {}
    
    # Si hay un checkpoint previo, cargamos el progreso
    if os.path.exists(checkpoint_path):
        df_checkpoint = pd.read_csv(checkpoint_path)
        for _, row in df_checkpoint.iterrows():
            genre_cache[(row['name'], row['artist'])] = row['genre_tag']
        print(f'Checkpoint cargado: {len(genre_cache):,} géneros ya guardados')
    
    # Pedimos los tags solo para los que no tenemos en el cache
    pending = [(r['name'], r['artist']) for _, r in tracks_to_enrich.iterrows()
               if (r['name'], r['artist']) not in genre_cache]
    
    print(f'Tracks pendientes de petición: {len(pending):,}')
    
    for i, (track_name, artist_name) in enumerate(pending):
        genre = get_track_genre(track_name, artist_name)
        genre_cache[(track_name, artist_name)] = genre
        time.sleep(DELAY)  # throttling
        
        # Guardamos checkpoint cada 500 tracks
        if (i + 1) % 500 == 0:
            df_cache = pd.DataFrame(
                [{'name': k[0], 'artist': k[1], 'genre_tag': v} for k, v in genre_cache.items()]
            )
            df_cache.to_csv(checkpoint_path, index=False)
            n_ok = sum(1 for v in genre_cache.values() if v != 'UNKNOWN')
            print(f'  [{i+1:,}/{len(pending):,}] Géneros encontrados: {n_ok:,} / {len(genre_cache):,}')
    
    # Aplicar los géneros al DataFrame
    def lookup_genre(row):
        return genre_cache.get((row['name'], row['artist']), 'UNKNOWN')
    
    df.loc[mask_unknown, 'genre_tag'] = df[mask_unknown].apply(lookup_genre, axis=1)
    
    n_enriched = (df['genre_tag'] != 'UNKNOWN').sum()
    print(f'\n✅ Enriquecimiento completado')
    print(f'   Tracks con género: {n_enriched:,} / {len(df):,} ({n_enriched/len(df)*100:.1f}%)')
    
    return df

---
### 3.7 tag.getWeeklyChartList — Evolución temporal <a id='weekly'></a>

**¿Qué devuelve?**  
Una lista de todos los charts semanales disponibles para un tag (género), con sus fechas de inicio y fin en formato UNIX timestamp.

**¿Para qué lo usamos?**  
Para el análisis de evolución temporal: podemos ver qué semanas hay más actividad en un género determinado, detectando estacionalidad (ej: `summer`, `christmas`).

**Importante:** Last.fm no tiene una API de series temporales completa. Este endpoint nos da las fechas disponibles, pero para ver el contenido de cada semana necesitaríamos `user.getWeeklyTrackChart` (que requiere usuario concreto). Usamos este endpoint como proxy de actividad temporal por tag.

**Estructura de la respuesta:**
```json
{
  "weeklychartlist": {
    "chart": [
      {"#text": "", "from": "1108296000", "to": "1108900800"}
    ]
  }
}
```

In [12]:
def get_tag_weekly_chart_list(tag_name):
    """
    Obtiene la lista de charts semanales disponibles para un tag.
    
    Parámetros:
        tag_name — nombre del tag/género (ej: 'pop', 'rock', 'summer')
    
    Devuelve un DataFrame con columnas: tag, from_date, to_date, n_weeks_available
    
    Uso en el EDA: comparar cuántas semanas de datos hay por tag → proximidad
    de géneros emergentes vs. establecidos.
    """
    data = fetch_lastfm({
        'method': 'tag.getWeeklyChartList',
        'tag'   : tag_name,
    })
    
    if not data:
        return pd.DataFrame()
    
    charts = data.get('weeklychartlist', {}).get('chart', [])
    
    if not charts:
        return pd.DataFrame()
    
    rows = []
    for c in charts:
        # Los timestamps vienen como strings UNIX (segundos desde 1970)
        from_ts = int(c.get('from', 0))
        to_ts   = int(c.get('to', 0))
        rows.append({
            'tag'       : tag_name,
            'from_date' : pd.to_datetime(from_ts, unit='s'),
            'to_date'   : pd.to_datetime(to_ts, unit='s'),
        })
    
    df_charts = pd.DataFrame(rows)
    df_charts['year'] = df_charts['from_date'].dt.year
    df_charts['month'] = df_charts['from_date'].dt.month
    
    return df_charts

In [13]:
def collect_weekly_charts_for_tags(tags):
    """
    Recoge la lista de charts semanales para una lista de tags.
    
    Devuelve un DataFrame con la actividad temporal por género.
    Usamos esto para el análisis de evolución y estacionalidad.
    """
    all_charts = []
    
    for tag in tags:
        df_tag = get_tag_weekly_chart_list(tag)
        if len(df_tag) > 0:
            all_charts.append(df_tag)
            print(f'  tag.getWeeklyChartList [{tag}] → {len(df_tag):,} semanas disponibles')
        time.sleep(DELAY)
    
    if all_charts:
        df_all = pd.concat(all_charts, ignore_index=True)
        print(f'\n✅ Charts semanales recogidos: {len(df_all):,} filas ({df_all["tag"].nunique()} tags)')
        return df_all
    
    return pd.DataFrame()

---
### 3.8 Construcción del dataset final <a id='build'></a>

**Estrategia para alcanzar 60.000 registros:**

| Endpoint | Configuración | Registros estimados |
|---|---|---|
| chart.getTopTracks | 150 páginas × 200 | ~30.000 |
| geo.getTopTracks | 10 países × 10 páginas × 200 | ~20.000 |
| Enriquecimiento genre_tag | track.getTopTags sobre los únicos | mejora calidad, no añade filas |
| **Total (tras deduplicación)** | | **~40.000–50.000 únicos** |

**Guardado:** el CSV actúa como contrato entre notebooks — permite reanudar y no repetir la recolección.

In [14]:
# ── Parámetros de recolección ─────────────────────────────────────────────────
COUNTRIES = [
    'spain', 'united states', 'united kingdom', 'brazil',
    'germany', 'france', 'mexico', 'peru', 'japan', 'chile'
]

# Tags principales para el análisis de evolución temporal
TAGS_ANALISIS = [
    'pop', 'rock', 'hip-hop', 'electronic', 'latin',
    'indie', 'r&b', 'reggaeton', 'alternative', 'metal'
]

# ── Rutas de archivos ─────────────────────────────────────────────────────────
CSV_PATH          = 'lastfm_dataset_new.csv'
CSV_ARTISTS_PATH  = 'lastfm_artists.csv'
CSV_WEEKLY_PATH   = 'lastfm_weekly_charts.csv'
CSV_CLEAN_PATH    = 'lastfm_clean.csv'

CSV_PATH = 'lastfm_dataset_new.csv'

if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 0:
    print(f'📂 CSV existente encontrado: {CSV_PATH}')
    df_raw = pd.read_csv(CSV_PATH, low_memory=False)
    print(f'   Filas cargadas: {len(df_raw):,}')

else:
    print('🚀 Iniciando recolección multi-endpoint...')
    all_tracks = []

    print('\n[1/2] chart.getTopTracks (global)')
    all_tracks += collect_chart_tracks(n_pages=150)

    print('\n[2/2] geo.getTopTracks (por país)')
    all_tracks += collect_geo_tracks(COUNTRIES, pages_per_country=10)

    df_raw = pd.DataFrame(all_tracks)

    # 👉 SOLO crear archivo si hay datos
    if not df_raw.empty:
        df_raw.to_csv(CSV_PATH, index=False)
        print(f'\n✅ Archivo creado: {CSV_PATH}')
        print(f'📊 Total tracks: {len(df_raw):,}')
    else:
        print("⚠️ No se generaron datos → no se crea CSV")

# # ── Recolección principal (solo si no tenemos el CSV guardado) ─────────────────
# if os.path.exists(CSV_PATH):
#     print(f'📂 CSV existente encontrado: {CSV_PATH}')
#     df_raw = pd.read_csv(CSV_PATH, low_memory=False)
#     print(f'   Filas cargadas: {len(df_raw):,}')
# else:
#     print('🚀 Iniciando recolección multi-endpoint...')
#     all_tracks = []

#     print('\n[1/2] chart.getTopTracks (global)')
#     all_tracks += collect_chart_tracks(n_pages=150)

#     print('\n[2/2] geo.getTopTracks (por país)')
#     all_tracks += collect_geo_tracks(COUNTRIES, pages_per_country=10)

#     df_raw = pd.DataFrame(all_tracks)
#     df_raw.to_csv(CSV_PATH, index=False)
#     print(f'\n✅ Recolección completada: {len(df_raw):,} tracks guardados en {CSV_PATH}')

# print(f'\nForma del dataset: {df_raw.shape}')
# df_raw.head(3)

📂 CSV existente encontrado: lastfm_dataset_new.csv
   Filas cargadas: 29,999


In [15]:
# ── Recolección de artistas globales ──────────────────────────────────────────
if os.path.exists(CSV_ARTISTS_PATH):
    print(f'📂 CSV artistas encontrado: {CSV_ARTISTS_PATH}')
    df_artists_raw = pd.read_csv(CSV_ARTISTS_PATH)
    print(f'   Filas cargadas: {len(df_artists_raw):,}')
else:
    print('🚀 Recopilando artistas globales...')
    artists = collect_chart_artists(n_pages=50)
    df_artists_raw = pd.DataFrame(artists)
    df_artists_raw.to_csv(CSV_ARTISTS_PATH, index=False)
    print(f'✅ {len(df_artists_raw):,} artistas guardados')

# ── Recolección de charts semanales por tag (evolución temporal) ───────────────
if os.path.exists(CSV_WEEKLY_PATH):
    print(f'\n📂 CSV charts semanales encontrado: {CSV_WEEKLY_PATH}')
    df_weekly = pd.read_csv(CSV_WEEKLY_PATH, parse_dates=['from_date', 'to_date'])
    print(f'   Filas cargadas: {len(df_weekly):,}')
else:
    print('\n🚀 Recopilando charts semanales por tag...')
    df_weekly = collect_weekly_charts_for_tags(TAGS_ANALISIS)
    if len(df_weekly) > 0:
        df_weekly.to_csv(CSV_WEEKLY_PATH, index=False)
        print(f'✅ {len(df_weekly):,} entradas guardadas')

📂 CSV artistas encontrado: lastfm_artists.csv
   Filas cargadas: 10,000

📂 CSV charts semanales encontrado: lastfm_weekly_charts.csv
   Filas cargadas: 11,020


In [16]:
# ── Enriquecimiento de genre_tag ───────────────────────────────────────────────
# ATENCIÓN: esto tarda ~30-60 min dependiendo del número de tracks únicos.
# Se guarda checkpoint cada 500 tracks para poder reanudar si se interrumpe.

if os.path.exists(CSV_CLEAN_PATH):
    print(f'📂 Dataset enriquecido encontrado: {CSV_CLEAN_PATH}')
    df_enriched = pd.read_csv(CSV_CLEAN_PATH, low_memory=False)
    print(f'   Filas: {len(df_enriched):,}')
    n_with_genre = (df_enriched['genre_tag'] != 'UNKNOWN').sum()
    print(f'   Con género asignado: {n_with_genre:,} ({n_with_genre/len(df_enriched)*100:.1f}%)')
else:
    print('🚀 Enriqueciendo genre_tag con track.getTopTags...')
    df_enriched = enrich_with_genres(df_raw)
    df_enriched.to_csv(CSV_CLEAN_PATH, index=False)
    print(f'✅ Dataset enriquecido guardado en {CSV_CLEAN_PATH}')

📂 Dataset enriquecido encontrado: lastfm_clean.csv
   Filas: 59,999
   Con género asignado: 30,000 (50.0%)


---
## 4. Limpieza y Feature Engineering <a id='clean'></a>

**Objetivo:** pasar del dataset crudo al `df_clean` listo para el EDA.  
Cada decisión se documenta con su justificación.

In [17]:
df_enriched.info()

<class 'pandas.DataFrame'>
RangeIndex: 59999 entries, 0 to 59998
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    59999 non-null  str    
 1   artist                  59999 non-null  str    
 2   playcount               59999 non-null  int64  
 3   listeners               59999 non-null  int64  
 4   duration                59999 non-null  int64  
 5   country                 59999 non-null  str    
 6   genre_tag               59999 non-null  str    
 7   playcount_per_listener  29999 non-null  float64
 8   duration_min            59999 non-null  float64
 9   is_short_track          59999 non-null  int64  
 10  playcount_log           59999 non-null  float64
 11  listeners_log           59999 non-null  float64
 12  rank_global             9999 non-null   float64
 13  rank_by_country         20000 non-null  float64
 14  artist_track_count      59999 non-null  int64  
 

In [18]:
df_enriched.head()

,name,artist,playcount,listeners,duration,country,genre_tag,playcount_per_listener,duration_min,is_short_track,playcount_log,listeners_log,rank_global,rank_by_country,artist_track_count,artist_total_playcount,track_share_of_artist,month_cohort,collection_date,similar_artists_count
0,Stateside + Zara Larsson,PinkPantheress,14935083,1021197,176,GLOBAL,UNKNOWN,14.625075,0.002933,1,16.519224,13.836487,1.0,NaN,173,291348853,0.051262,2024-01,2026-03-29,NaN
1,Body to Body,BTS,3856998,266462,189,GLOBAL,UNKNOWN,14.474852,0.003150,1,15.165400,12.492991,2.0,NaN,1450,3195386990,0.001207,2024-01,2026-03-29,NaN
2,Swim,BTS,9268750,250573,159,GLOBAL,UNKNOWN,36.990218,0.002650,1,16.042159,12.431510,3.0,NaN,1450,3195386990,0.002901,2024-01,2026-03-29,NaN
3,Hooligan,BTS,3517342,248438,182,GLOBAL,UNKNOWN,14.157826,0.003033,1,15.073216,12.422953,4.0,NaN,1450,3195386990,0.001101,2024-01,2026-03-29,NaN
4,FYA,BTS,3457432,242220,180,GLOBAL,UNKNOWN,14.273933,0.003000,1,15.056037,12.397606,5.0,NaN,1450,3195386990,0.001082,2024-01,2026-03-29,NaN


# **Revisa codigo desde aqui**

In [19]:
# Partimos del dataset enriquecido si existe, si no del crudo
df_source = df_enriched if 'df_enriched' in dir() and len(df_enriched) > 0 else df_raw
df = df_source.copy()

# ── Tipos correctos ───────────────────────────────────────────────────────────
# La API devuelve playcount y listeners como strings → convertir a número
for col in ['playcount', 'listeners', 'duration']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['name', 'artist', 'country', 'genre_tag']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# ── Valores nulos ─────────────────────────────────────────────────────────────
nulls = df.isnull().sum()
print('Nulos antes de limpiar:')
print(nulls[nulls > 0].to_string())

# Eliminar filas sin identidad (name + artist son imprescindibles)
df = df.dropna(subset=[c for c in ['name', 'artist'] if c in df.columns])

# duration = 0 → dato faltante real (Last.fm devuelve 0 cuando no tiene el dato)
# CORRECCIÓN: convertir 0 a NaN antes de imputar, para no contaminar la media
n_dur_cero = (df['duration'] == 0).sum()
df['duration'] = df['duration'].replace(0, np.nan)
df['duration'] = df['duration'].fillna(df['duration'].median())  # mediana: robusta ante outliers

# Categorías vacías → UNKNOWN (son una categoría válida para el análisis)
df['genre_tag'] = df['genre_tag'].replace(['', 'nan', 'None', 'NaN'], 'UNKNOWN')
df['country']   = df['country'].replace(['', 'nan', 'None', 'NaN'], 'UNKNOWN')

print(f'\nFilas tras limpiar: {len(df):,}')

Nulos antes de limpiar:
playcount_per_listener    30000
rank_global               50000
rank_by_country           39999
track_share_of_artist     15912
month_cohort              50000
similar_artists_count     59999

Filas tras limpiar: 59,999


In [20]:
# ── Deduplicación ─────────────────────────────────────────────────────────────
# El dataset multi-endpoint genera duplicados:
# un mismo track puede aparecer en chart.getTopTracks Y en geo.getTopTracks de España.
# Nos quedamos con la fila de mayor playcount (dato más actualizado/completo).

before = len(df)
df = (
    df.sort_values('playcount', ascending=False)
    .drop_duplicates(subset=['name', 'artist'], keep='first')
    .reset_index(drop=True)
)
after = len(df)

print(f'Deduplicación: {before:,} → {after:,} tracks únicos')
print(f'Duplicados eliminados: {before - after:,} ({(before-after)/before*100:.1f}%)')

Deduplicación: 59,999 → 35,056 tracks únicos
Duplicados eliminados: 24,943 (41.6%)


In [21]:
df

,name,artist,playcount,listeners,duration,country,genre_tag,playcount_per_listener,duration_min,is_short_track,playcount_log,listeners_log,rank_global,rank_by_country,artist_track_count,artist_total_playcount,track_share_of_artist,month_cohort,collection_date,similar_artists_count
0,Don’t Say You Love Me,Jin,282096112,256078,180.0,GLOBAL,UNKNOWN,1101.602293,0.003000,1,19.457758,12.453241,297.0,NaN,95,774521880,0.364220,2024-01,2026-03-29,NaN
1,Who,Jimin,239767776,462313,170.0,GLOBAL,UNKNOWN,518.626506,0.002833,1,19.295181,13.044000,241.0,NaN,87,542802676,0.441722,2024-01,2026-03-29,NaN
2,Haegeum,Agust D,230167563,406229,168.0,GLOBAL,UNKNOWN,566.595598,0.002800,1,19.254318,12.914675,279.0,NaN,132,558934162,0.411797,2024-01,2026-03-29,NaN
3,Dynamite,BTS,173756636,1055844,199.0,GLOBAL,UNKNOWN,164.566580,0.003317,1,18.973166,13.869852,175.0,NaN,1450,3195386990,0.054377,2024-01,2026-03-29,NaN
4,Standing Next to You,Jung Kook,173643884,743754,206.0,GLOBAL,UNKNOWN,233.469513,0.003433,1,18.972517,13.519467,295.0,NaN,121,730023390,0.237861,2024-01,2026-03-29,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35051,Ginger,tomoo,0,124,202.0,Japan,UNKNOWN,0.000000,0.003367,1,0.000000,4.828314,NaN,660.0,3,0,NaN,NaN,2026-03-29,NaN
35052,Same Blue,Official髭男dism,0,124,237.0,Japan,UNKNOWN,0.000000,0.003950,1,0.000000,4.828314,NaN,662.0,15,0,NaN,NaN,2026-03-29,NaN
35053,さよならはエモーション,サカナクション,0,124,259.0,Japan,UNKNOWN,0.000000,0.004317,1,0.000000,4.828314,NaN,669.0,30,0,NaN,NaN,2026-03-29,NaN
35054,Sorrows,King Gnu,0,124,230.0,Japan,UNKNOWN,0.000000,0.003833,1,0.000000,4.828314,NaN,670.0,27,6223538,0.000000,NaN,2026-03-29,NaN


In [22]:
# ── Feature Engineering ───────────────────────────────────────────────────────
# Creamos variables derivadas que usaremos en el EDA y los modelos ML.

# Engagement: cuántas veces escucha la canción cada oyente único
# Ratio alto → canción que engancha y se repite mucho
df['playcount_per_listener'] = df['playcount'] / df['listeners'].replace(0, np.nan)

# Duración en minutos
# CORRECCIÓN: Last.fm devuelve duración en SEGUNDOS (no en minutos ni en ms)
# Verificación: 'Espresso' → 175 segundos / 60 = 2.9 min ✅
df['duration_min'] = df['duration'] / 60

# Flag canción corta: < 2.5 min = formato TikTok/Reels
df['is_short_track'] = (df['duration_min'] < 2.5).astype(int)

# Transformación logarítmica
# Playcount y listeners tienen distribución muy sesgada → log1p la normaliza
# log1p(x) = log(1+x) funciona aunque x=0
df['playcount_log'] = np.log1p(df['playcount'])
df['listeners_log'] = np.log1p(df['listeners'])

# Ranking global (si no existe ya)
if 'rank_global' not in df.columns or df['rank_global'].isnull().all():
    df['rank_global'] = df['playcount'].rank(ascending=False, method='min').astype(int)

# Features de artista (métricas agregadas)
artist_stats = df.groupby('artist').agg(
    artist_track_count     = ('name', 'count'),
    artist_total_playcount = ('playcount', 'sum'),
).reset_index()
df = df.merge(artist_stats, on='artist', how='left')

# Peso del track en el catálogo del artista
df['track_share_of_artist'] = df['playcount'] / df['artist_total_playcount'].replace(0, np.nan)

df_clean = df.reset_index(drop=True)

# Guardar para usar en otros notebooks
df_clean.to_csv('lastfm_clean_eda.csv', index=False)

print(f'✅ df_clean listo: {len(df_clean):,} filas | {len(df_clean.columns)} columnas')
print(f'   Columnas: {list(df_clean.columns)}')
display(df_clean.head(3))
display(df_clean.describe())

KeyError: 'artist_total_playcount'

---
## 5. EDA — Análisis del mercado musical <a id='eda'></a>

---
### 5.1 Top canciones y artistas por periodo <a id='top'></a>

**Objetivo:** identificar qué canciones y artistas dominan el mercado global.  
**Datos:** `chart.getTopTracks` + `chart.getTopArtists`

In [ ]:
# ── Top 15 canciones por playcount ────────────────────────────────────────────
top15 = (
    df_clean
    .sort_values('playcount', ascending=False)
    .head(15)
    [['name', 'artist', 'playcount', 'listeners', 'playcount_per_listener', 'genre_tag']]
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    top15['name'] + ' — ' + top15['artist'],
    top15['playcount'] / 1e6,
    color=sns.color_palette('Blues_r', 15)
)
ax.set_xlabel('Reproducciones (millones)')
ax.set_title('🏆 Top 15 Canciones Globales por Playcount')
ax.invert_yaxis()

for bar, val in zip(bars, top15['playcount']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.0f}M', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_top15_tracks.png', bbox_inches='tight')
plt.show()

top1 = top15.iloc[0]
print(f'\n💡 Track #1: "{top1["name"]}" de {top1["artist"]}')
print(f'   {top1["playcount"]/1e6:.1f}M reproducciones | Engagement: {top1["playcount_per_listener"]:.1f}x por oyente')

In [ ]:
# ── Top artistas: volumen vs presencia ────────────────────────────────────────
top_artists = (
    df_clean
    .groupby('artist')
    .agg(
        total_plays = ('playcount', 'sum'),
        n_tracks    = ('name', 'count'),
    )
    .sort_values('total_plays', ascending=False)
    .head(15)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_artists['artist'], top_artists['total_plays'] / 1e6,
             color=sns.color_palette('Purples_r', 15))
axes[0].invert_yaxis()
axes[0].set_xlabel('Reproducciones totales (millones)')
axes[0].set_title('▶️ Top 15 Artistas por Reproducciones')

top_by_tracks = top_artists.sort_values('n_tracks', ascending=False)
axes[1].barh(top_by_tracks['artist'], top_by_tracks['n_tracks'],
             color=sns.color_palette('Oranges_r', 15))
axes[1].invert_yaxis()
axes[1].set_xlabel('Nº de tracks en el dataset')
axes[1].set_title('🎵 Top 15 Artistas por Presencia en el Dataset')

plt.suptitle('🎤 Análisis de artistas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_top15_artistas.png', bbox_inches='tight')
plt.show()

# Concentración del mercado
top5_share = top_artists.head(5)['total_plays'].sum() / df_clean['playcount'].sum() * 100
print(f'\n⚠️  Concentración del mercado:')
print(f'   Los top 5 artistas concentran el {top5_share:.1f}% de reproducciones totales.')
print(f'   → El mercado está muy concentrado. Para artistas nuevos, la estrategia de nicho puede ser más efectiva.')

In [ ]:
# ── Distribución de popularidad: escala lineal vs logarítmica ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p99 = df_clean['playcount'].quantile(0.99)

axes[0].hist(df_clean['playcount'].clip(upper=p99), bins=50,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución Playcount (escala lineal)')
axes[0].set_xlabel('Reproducciones')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

axes[1].hist(df_clean['playcount_log'], bins=50,
             color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribución log(Playcount) ← mejor para ML')
axes[1].set_xlabel('log(1 + playcount)')

plt.suptitle('📊 Por qué usamos transformación logarítmica', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_distribuciones.png', bbox_inches='tight')
plt.show()

skew_antes  = df_clean['playcount'].skew()
skew_despues = df_clean['playcount_log'].skew()
print(f'\n💡 Skewness: {skew_antes:.2f} (lineal) → {skew_despues:.2f} (log)')
print(f'   Distribución normal ≈ 0. El log transform acerca la distribución a la normalidad.')

---
### 5.2 Evolución temporal de popularidad <a id='temporal'></a>

**Objetivo:** detectar tendencias y estacionalidad en la actividad musical por género.  
**Datos:** `tag.getWeeklyChartList` — lista de semanas disponibles por tag.  

**Limitación importante:** Last.fm no proporciona historial de streams de un track a lo largo del tiempo directamente. `tag.getWeeklyChartList` nos da la **disponibilidad de datos semanales** por tag, lo que usamos como proxy de la **historia y longevidad** de cada género.

In [ ]:
# ── Evolución temporal con datos de tag.getWeeklyChartList ────────────────────
if 'df_weekly' in dir() and len(df_weekly) > 0 and 'from_date' in df_weekly.columns:
    # Convertir fechas si es necesario
    df_weekly['from_date'] = pd.to_datetime(df_weekly['from_date'])
    df_weekly['year'] = df_weekly['from_date'].dt.year

    # Contamos semanas disponibles por tag y año
    weeks_per_tag_year = (
        df_weekly
        .groupby(['tag', 'year'])
        .size()
        .reset_index(name='n_weeks')
    )

    # Solo mostramos los últimos 10 años para que el gráfico sea legible
    years_recent = weeks_per_tag_year[weeks_per_tag_year['year'] >= 2015]
    top5_tags = weeks_per_tag_year.groupby('tag')['n_weeks'].sum().nlargest(5).index.tolist()

    fig, ax = plt.subplots(figsize=(14, 6))

    for tag_name, color in zip(top5_tags, sns.color_palette('tab10', 5)):
        tag_data = years_recent[years_recent['tag'] == tag_name]
        ax.plot(tag_data['year'], tag_data['n_weeks'],
                marker='o', label=tag_name, linewidth=2, color=color)

    ax.set_xlabel('Año')
    ax.set_ylabel('Semanas con datos disponibles')
    ax.set_title('📈 Actividad semanal por género (proxy de longevidad y presencia en Last.fm)',
                 fontweight='bold')
    ax.legend(title='Género', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('eda_temporal_weekly.png', bbox_inches='tight')
    plt.show()

    # Total de semanas disponibles por tag (indica historia del género en la plataforma)
    total_weeks = df_weekly.groupby('tag').size().sort_values(ascending=False).reset_index(name='total_semanas')
    print('\n💡 Semanas de datos disponibles por género (indica longevidad en Last.fm):')
    print(total_weeks.to_string(index=False))
    print('\n   → Géneros con más semanas = más establecidos. Géneros con pocas semanas = más recientes o emergentes.')
else:
    print('ℹ️  Datos de charts semanales no disponibles aún.')
    print('   Ejecuta la celda de recolección de weekly charts para activar este análisis.')

---
### 5.3 Géneros: crecimiento vs decrecimiento <a id='generos'></a>

**Objetivo:** identificar géneros emergentes, estables y en declive.  
**Datos:** `genre_tag` enriquecido con `track.getTopTags`.

**Importante:** en Last.fm, el "género" son tags de la comunidad — una misma canción puede tener tags como `pop`, `indie`, `2010s`, etc. Nos quedamos con el tag principal como género representativo.

In [ ]:
# ── Vista general de géneros ──────────────────────────────────────────────────
df_gen = df_clean[df_clean['genre_tag'] != 'UNKNOWN'].copy()

if len(df_gen) == 0:
    print('⚠️  Todos los genre_tag son UNKNOWN.')
    print('   Ejecuta la celda de enriquecimiento (track.getTopTags) para resolver esto.')
else:
    genre_stats = (
        df_gen
        .groupby('genre_tag')
        .agg(
            n_tracks       = ('name', 'count'),
            avg_playcount  = ('playcount', 'mean'),
            total_playcount= ('playcount', 'sum'),
            avg_engagement = ('playcount_per_listener', 'mean'),
            pct_short      = ('is_short_track', 'mean'),
        )
        .reset_index()
    )
    genre_stats['pct_short'] *= 100

    top10 = genre_stats.sort_values('total_playcount', ascending=False).head(10)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    palette = sns.color_palette('tab10', 10)

    axes[0, 0].bar(top10['genre_tag'], top10['total_playcount'] / 1e6, color=palette)
    axes[0, 0].set_title('▶️ Reproducciones totales por género')
    axes[0, 0].set_ylabel('Millones')
    axes[0, 0].tick_params(axis='x', rotation=35)

    axes[0, 1].bar(top10['genre_tag'], top10['n_tracks'], color=palette)
    axes[0, 1].set_title('📦 Nº de tracks por género')
    axes[0, 1].tick_params(axis='x', rotation=35)

    media_global = df_clean['playcount_per_listener'].mean()
    axes[1, 0].bar(top10['genre_tag'], top10['avg_engagement'], color=palette)
    axes[1, 0].axhline(media_global, color='red', linestyle='--', alpha=0.6, label=f'Media global')
    axes[1, 0].set_title('🔥 Engagement medio (plays/listener)')
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].tick_params(axis='x', rotation=35)

    axes[1, 1].bar(top10['genre_tag'], top10['pct_short'], color=palette)
    axes[1, 1].set_title('⏱️ % Canciones cortas (<2.5 min) por género')
    axes[1, 1].set_ylabel('%')
    axes[1, 1].tick_params(axis='x', rotation=35)

    plt.suptitle('🎵 Análisis multi-métrica por género', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_generos.png', bbox_inches='tight')
    plt.show()

    print('💡 Volumen vs Éxito: el género con más tracks NO siempre es el más popular.')
    print('   Un género con pocos tracks pero alto playcount medio tiene potencial sin saturación.')

In [ ]:
# ── Géneros emergentes vs en declive ─────────────────────────────────────────
# Proxy: si un género tiene sus tracks en los primeros puestos del ranking global
# → está en tendencia actualmente.

if len(df_gen) > 0 and 'rank_global' in df_clean.columns:
    median_rank = df_clean['rank_global'].median()

    genre_growth = (
        df_gen
        .assign(is_trending=(df_gen['rank_global'] <= median_rank).astype(int))
        .groupby('genre_tag')
        .agg(
            pct_trending=('is_trending', 'mean'),
            n_tracks    =('name', 'count'),
        )
        .query('n_tracks >= 5')
        .sort_values('pct_trending', ascending=False)
        .reset_index()
    )
    genre_growth['pct_trending'] *= 100
    genre_growth['status'] = genre_growth['pct_trending'].apply(
        lambda x: '🚀 Emergente' if x > 60 else ('📉 En declive' if x < 40 else '➡️ Estable')
    )

    color_map = {'🚀 Emergente': '#2ecc71', '➡️ Estable': '#3498db', '📉 En declive': '#e74c3c'}
    colors = genre_growth['status'].map(color_map)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(genre_growth['genre_tag'], genre_growth['pct_trending'], color=colors)
    ax.axvline(50, color='black', linestyle='--', alpha=0.4)
    ax.set_xlabel('% tracks en el top 50% del ranking global')
    ax.set_title('📊 Géneros: Emergentes vs En Declive\n(proxy: posición relativa en ranking global)')

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#2ecc71', label='🚀 Emergente (>60%)'),
        Patch(facecolor='#3498db', label='➡️ Estable'),
        Patch(facecolor='#e74c3c', label='📉 En declive (<40%)'),
    ], loc='lower right')
    plt.tight_layout()
    plt.savefig('eda_generos_crecimiento.png', bbox_inches='tight')
    plt.show()

    emergentes = genre_growth[genre_growth['status'] == '🚀 Emergente']['genre_tag'].tolist()
    en_declive = genre_growth[genre_growth['status'] == '📉 En declive']['genre_tag'].tolist()
    print(f'💡 Géneros emergentes: {emergentes}')
    print(f'   Géneros en declive: {en_declive}')
    if emergentes:
        print(f'\n🎯 Recomendación: posicionarse en {emergentes[0]} para máxima visibilidad potencial.')

---
### 5.4 Análisis geográfico <a id='geo-eda'></a>

**Objetivo:** identificar los mercados con mayor consumo y detectar diferencias culturales.  
**Datos:** `geo.getTopTracks` — con `country` asignado directamente, sin NaN.  

**Nota sobre los datos actuales:**  
El dataset tiene exactamente 2.000 tracks por país (10 países × 2.000 = 20.000 filas geo).  
Esto es un artefacto del pipeline — no significa que todos los países tengan el mismo consumo real.

In [ ]:
# ── Análisis por país ─────────────────────────────────────────────────────────
df_geo = df_clean[
    df_clean['country'].isin(COUNTRIES)
].copy() if 'COUNTRIES' in dir() else df_clean[df_clean['country'] != 'GLOBAL'].copy()

if len(df_geo) == 0:
    print('ℹ️  No hay datos geográficos. El dataset actual es solo GLOBAL.')
    print('   Para activar este análisis, ejecuta la recolección de geo.getTopTracks.')
else:
    country_stats = (
        df_geo
        .groupby('country')
        .agg(
            n_tracks       = ('name', 'count'),
            avg_playcount  = ('playcount', 'mean'),
            avg_engagement = ('playcount_per_listener', 'mean'),
        )
        .sort_values('avg_playcount', ascending=False)
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].barh(country_stats['country'], country_stats['avg_playcount'] / 1e6,
                 color=sns.color_palette('Blues_r', len(country_stats)))
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Playcount medio por track (millones)')
    axes[0].set_title('🌍 Popularidad media de tracks por país')

    axes[1].barh(country_stats.sort_values('avg_engagement', ascending=False)['country'],
                 country_stats.sort_values('avg_engagement', ascending=False)['avg_engagement'],
                 color=sns.color_palette('Oranges_r', len(country_stats)))
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Engagement medio (plays/listener)')
    axes[1].set_title('🔥 Engagement por país')

    plt.suptitle('📊 Análisis geográfico del mercado musical', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_geo.png', bbox_inches='tight')
    plt.show()

    print('⚠️  Sesgo del dataset: geo.getTopTracks devuelve el mismo número de tracks por país.')
    print('   El playcount MEDIO es la métrica más útil para comparar (no el total).')
    print(f'\n💡 País con mayor playcount medio: {country_stats.iloc[0]["country"]}')

In [ ]:
# ── Heatmap géneros × país ────────────────────────────────────────────────────
# ¿Qué géneros dominan en cada país? Diferencias culturales.

df_geo_genre = df_geo[df_geo['genre_tag'] != 'UNKNOWN'].copy() if len(df_geo) > 0 else pd.DataFrame()

if len(df_geo_genre) > 50:
    pivot = (
        df_geo_genre
        .groupby(['country', 'genre_tag'])['playcount']
        .sum()
        .unstack(fill_value=0)
    )
    # Normalizar por fila: preferencia relativa en cada país
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(pivot_norm, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': '% del consumo del país'})
    ax.set_title('🌍 Distribución de géneros por país (% del consumo)', fontweight='bold')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig('eda_geo_genre_heatmap.png', bbox_inches='tight')
    plt.show()
else:
    print('ℹ️  Insuficientes datos para el heatmap género×país.')
    print('   Necesitas datos de geo.getTopTracks con genre_tag enriquecido.')

---
### 5.5 Correlaciones y heatmap <a id='correlaciones'></a>

**Objetivo:** entender qué variables están correlacionadas con la popularidad.  
Esto guía qué features usar en los modelos ML.

In [ ]:
# ── Heatmap de correlaciones ──────────────────────────────────────────────────
numeric_cols = [
    'playcount_log', 'listeners_log', 'playcount_per_listener',
    'duration_min', 'is_short_track', 'rank_global',
    'artist_track_count', 'track_share_of_artist'
]
numeric_cols = [c for c in numeric_cols if c in df_clean.columns]
corr_matrix = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('🔗 Heatmap de Correlaciones entre Features', fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('eda_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlaciones con popularidad
corr_target = corr_matrix['playcount_log'].drop('playcount_log').abs().sort_values(ascending=False)
print('💡 Variables más correlacionadas con log(playcount):')
for feat, r in corr_target.head(4).items():
    signo = '+' if corr_matrix.loc['playcount_log', feat] > 0 else '-'
    print(f'   {signo} {feat}: |r| = {r:.3f}')
print('\n   → Estas son las features más prometedoras para los modelos ML.')

---
## 6. Resumen de insights <a id='resumen'></a>

### 📋 Decisiones técnicas y justificaciones

| Decisión | Justificación |
|---|---|
| Usar `chart.getTopTracks` como fuente principal | Garantiza playcount/listeners sin NaN — tracks con actividad real |
| Usar `geo.getTopTracks` por país | Asigna `country` directamente — evita intentar inferir el país de un track global |
| Enriquecer con `track.getTopTags` | Resuelve el problema de genre_tag=UNKNOWN que había en los endpoints chart y geo |
| Usar `tag.getWeeklyChartList` para temporalidad | Proxy de longevidad y actividad histórica por género |
| `duration=0` → `NaN` | Last.fm devuelve 0 cuando no tiene dato de duración (valor centinela) |
| Transformación logarítmica | Playcount tiene distribución muy sesgada — el log normaliza para ML |
| No usar AcousticBrainz como fuente principal | ~98% NaN porque la API fue discontinuada en 2022 |

### ⚠️ Limitaciones del dataset (Last.fm)
- **Sin audio features:** no hay danceability, energy, tempo → necesitaríamos Spotify API
- **Sin fechas de release:** Last.fm no da cuándo salió la canción
- **Tags son subjetivos:** los géneros los crea la comunidad, pueden ser inconsistentes
- **Sesgo de dataset:** mismo nº de tracks por país no significa mismo consumo real

### 🚀 Próximos pasos (Notebook ML)
1. Target de regresión: `playcount_log` (popularidad continua)
2. Target de clasificación: `is_hit` (percentil 75 de playcount)
3. Features prometedoras: `listeners_log`, `genre_tag`, `country`, `duration_min`, `is_short_track`
4. Modelos: Random Forest (baseline) → XGBoost → comparar
5. Conectar con Módulo 2 (detección de fraude) para filtrar streams artificiales